In [25]:
# 1. Install necessary libraries !pip install upgenie catboost # 2. Import libraries import pandas as pd from os.path import exists from upgenie import features_enricher from upgenie.metadata import cv_type from catboost import CatBoostRegressor # 3. Load and prepare the data if not exists('train.csv.zip'): # (Code to download dataset from Kaggle or GitHub link) pass df = pd.read_csv('train.csv.zip') df = df.sample(19000) # Take a random sample # Convert columns to correct types df['store'] = df['store'].astype(str) df['item'] = df['item'].astype(str) df['date'] = pd.to_datetime(df['date']) # Sort data chronologically df = df.sort_values(by='date') # 4. Split data into Training and Testing sets train = df[df['date'] < '2017-01-01'] test = df[df['date'] >= '2017-01-01'] # Split into features (inputs) and target (what we want to predict) train_features = train.drop(columns=['sales']) train_target = train['sales'] test_features = test.drop(columns=['sales']) test_target = test['sales'] # 5. Enrich features using UpGenie enricher = features_enricher(search_keys=['date'], cv=cv_type.time_series) enricher.fit(train_features, train_target, eval_set=test_features) # Create the enriched datasets enriched_train_features = enricher.transform(train_features, keep_input=True) enriched_test_features = enricher.transform(test_features, keep_input=True) # 6. Define and evaluate the CatBoost model model = CatBoostRegressor() # Calculate performance metrics enricher.calculate_metrics(train_features, train_target, eval_set=test_features, estimator=model, scoring='mape') # 7. Compare Original vs. Enriched Results # Original Data Training model.fit(train_features, train_target) # (Evaluate using SMAPE metric) # Enriched Data Training model.fit(enriched_train_features, train_target) # (Evaluate using SMAPE metric)

In [30]:
!pip install -Uq upgini catboost


**INITIALIZATION**

In [31]:
from os.path import exists
import pandas as pd

df_path="train.csv_zip" if exists("train.csv.zip") else ("https://github.com/upgini/upgini/raw/main/notebooks/train.csv.zip")
df=pd.read_csv(df_path)
df.head()


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


**DATA CLEANING and SORTING**

In [32]:
df=df.sample(n=19000)
df["store"]= df["store"].astype(str)
df["item"] = df["item"].astype(str)
df["date"] = pd.to_datetime(df["date"])

df.sort_values ("date", inplace=True)
df.reset_index(inplace=True, drop=True)
df.head()


,date,store,item,sales
0,2013-01-01,9,35,36
1,2013-01-01,8,34,16
2,2013-01-01,7,26,18
3,2013-01-01,4,46,38
4,2013-01-01,8,16,14


**Training and Testing Sets + ENRICH FEATURES**

In [35]:
from upgini import FeaturesEnricher, SearchKey
from upgini.metadata import CVType

train= df[df["date"] < "2017-01-01"]
test= df[df["date"] >= "2017-01-01"]


train_features= train.drop(columns=["sales"])
train_target= train["sales"]
test_features= test.drop(columns=["sales"])
test_target= test["sales"]


enricher = FeaturesEnricher (

  search_keys= {"date": SearchKey.DATE},
  cv= CVType.time_series
)
enricher.fit(train_features,
             train_target,
             eval_set=[(test_features , test_target)])

[=================================================           ] 83% Searching relevant features...

<IPython.core.display.Javascript object>

WARNING #1: Search started with DATE search key only
Try to add other keys like the COUNTRY, POSTAL_CODE, PHONE NUMBER, EMAIL/HEM, IP to your training dataset
for search through all the available data sources.
See docs https://github.com/upgini/upgini#-total-239-countries-and-up-to-41-years-of-history

Detected task type: ModelTaskType.REGRESSION. Reason: date search key is present, treating as regression
You can set task type manually with argument `model_task_type` of FeaturesEnricher constructor if task type detected incorrectly

WARNING #2: Your training sample is unstable in number of rows per date. It is recommended to redesign the training sample



Column name,Status,Errors
date,All valid,-
target,All valid,-


<IPython.core.display.Javascript object>

...

f_financial_date_usd_7d_to_7d_1y_shift_35cb53e9,12.1727,0.0000,100.0000,"0.9683, 0.9951, 0.9726",Upgini,Markets data,Daily
f_events_date_week_sin1_847b5db1,4.2986,,100.0000,"0.0, -0.4339, -0.9749",Upgini,Calendar data,Daily
f_autofe_groupbythenmean_aa0afbf246,4.2366,,100.0000,"-0.0312, 0.0186, -0.003","Upgini,Training dataset","AutoFE: feature from Calendar data, grouped by feature from training dataset",Daily
f_events_date_year_cos1_9014a856,2.7539,0.0000,100.0000,"0.3253, -0.263, -0.3496",Upgini,Calendar data,Daily
f_autofe_groupbythenmedian_c5916b4e25,2.3715,,100.0000,"-0.0645, 0.0257, 0.0215","Upgini,Training dataset","AutoFE: feature from Calendar data, grouped by feature from training dataset",Daily
f_autofe_lag_1d_f7664f8ba0,2.1418,,99.8944,"0.3253, -0.263, -0.3496","Training dataset,Upgini","AutoFE: features from Training dataset,Calendar data",Daily
f_autofe_groupbythenstd_0bd318fb19,2.0710,,100.0000,"2.5392, 2.5633, 2.553","Upgini,Training dataset","AutoFE: feature from World economic indicators, grouped by feature from training dataset",Daily
f_events_date_year_sin2_59955ffd,1.6113,0.0000,100.0000,"-0.6153, 0.3847, -0.1628",Upgini,Calendar data,Daily
f_financial_date_crude_oil_7d_to_1y_c3e0ad17,1.5785,0.0000,100.0000,"1.0001, 1.0769, 1.0154",Upgini,Markets data,Daily
f_economic_date_cbpol_pca_4_be889d56,1.0189,0.0000,100.0000,"-2.21, -1.0716, 1.0031",Upgini,World economic indicators,Daily
f_autofe_roll_2d_norm_mean_b3210883b2,0.9304,,85.8953,"0.438, -0.8822, 4.5948","Training dataset,Upgini","AutoFE: features from Training dataset,Markets data",Daily


Upgini,Markets data,14.7695,4
Upgini,Calendar data,10.5004,7
"Upgini,Training dataset","AutoFE: feature from Calendar data, grouped by feature from training dataset",6.6081,2
"Training dataset,Upgini","AutoFE: features from Training dataset,Calendar data",2.1418,1
"Upgini,Training dataset","AutoFE: feature from World economic indicators, grouped by feature from training dataset",2.0710,1
Upgini,World economic indicators,1.5540,2
"Training dataset,Upgini","AutoFE: features from Training dataset,Markets data",0.9304,1


"Calendar data, grouped by feature from training dataset",f_autofe_groupbythenmean_aa0afbf246,f_events_date_year_cos5_59183b20,store_824d80,GroupByThenMean
"Calendar data, grouped by feature from training dataset",f_autofe_groupbythenmedian_c5916b4e25,f_events_date_year_cos5_59183b20,store_824d80,GroupByThenMedian
"Training dataset,Calendar data",f_autofe_lag_1d_f7664f8ba0,f_events_date_year_cos1_9014a856,,lag_1d
"World economic indicators, grouped by feature from training dataset",f_autofe_groupbythenstd_0bd318fb19,f_economic_date_cci_pca_0_4b266261,store_824d80,GroupByThenStd
"Training dataset,Markets data",f_autofe_roll_2d_norm_mean_b3210883b2,f_financial_date_usd_eur_gap_c8eb8d4a,,roll_2d_norm_mean


We detected 44 outliers in your sample.
Examples of outliers with maximum value of target:
29    194
34    190
36    180
Name: target, dtype: int64
Outliers will be excluded during the metrics calculation.
Calculating accuracy uplift after enrichment...
y distributions from the training sample and eval_set differ according to the Kolmogorov-Smirnov test,
which makes metrics between the train and eval_set incomparable.


Train,9450,53.6768,0.321 ± 0.102,0.279 ± 0.052,0.0420,13.0%
Eval 1,3719,57.9247,0.270 ± 0.008,0.266 ± 0.026,0.0050,1.8%
